# 02 — Sales trends

Monthly revenue, order volume, average order value, seasonality.

**The trap (§9).** The dataset ends part-way through October 2018, so the final
months show an apparent collapse in revenue that is an artifact of truncation,
not a business event. Every monthly chart must exclude or annotate the
incomplete tail — `var('last_complete_month')` in `dbt_project.yml` marks the
cut. Presenting that decline to an executive audience as a finding is the most
likely way to lose credibility in Q&A.

Expect Black Friday 2017 (late November) to be the clearest genuine seasonal
signal in the series.

Owner: lane B2.


In [ ]:
import os

import pandas as pd
import numpy as np

# from dotenv import load_dotenv
# from sqlalchemy import create_engine

# load_dotenv("../.env")

# # Marts only. The dialect keeps the warehouse swappable (§9).
# engine = create_engine(f"bigquery://{os.environ['GCP_PROJECT']}/olist_marts")
data_path=[]

for dirname, _, filenames in os.walk('/home/bwong/M2/ntu-dsai-group5-project2/data/staging/'):
    for filename in filenames:
       data_path.append(os.path.join(dirname, filename))


data_path

In [ ]:
customers=data_path[0]
sellers=data_path[1]
review=data_path[2]
items=data_path[3]
products=data_path[4]
geolocation=data_path[5]
category_name_translation=data_path[6]
orders=data_path[7]
order_payments=data_path[8]

In [ ]:
customers = pd.read_csv(customers)
sellers = pd.read_csv(sellers)
review = pd.read_csv(review)
items = pd.read_csv(items)
products = pd.read_csv(products)
geolocation = pd.read_csv(geolocation)
orders = pd.read_csv(orders)
order_payments = pd.read_csv(order_payments)

In [ ]:
customers.head(10)

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

# Define target BigQuery details
project_id = "dsai6mod2"
dataset_id = "dsdwh"
table_id = "sales_trends"

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

orders['org_to_carrier_time'] = orders['order_delivered_carrier_date'] - orders['order_approved_at']
orders['carrier_to_customer_time'] = orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']
orders['actual_arrival_time'] = orders['order_delivered_customer_date'] - orders['order_approved_at']
orders['apply_SLA'] = orders['order_delivered_customer_date'] <= orders['order_estimated_delivery_date']
orders['delivery_delay']= orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']

orders_sla = orders[orders['apply_SLA'] == True]

order_items=pd.merge(orders_sla,items,on='order_id')

order_items["order_purchase_timestamp"]=pd.to_datetime(order_items["order_purchase_timestamp"])
order_items['month']=order_items["order_purchase_timestamp"].dt.month
order_items['year']=order_items["order_purchase_timestamp"].dt.year

# Option 1: Using pandas-gbq
order_items.to_gbq(
    destination_table=f"{dataset_id}.{table_id}",
    project_id=project_id,
    if_exists="replace"  # options: 'fail', 'replace', 'append'
)
    
print("DataFrame loaded successfully into BigQuery!")


In [ ]:
order_items["order_purchase_timestamp"]=pd.to_datetime(order_items["order_purchase_timestamp"])
order_items['month']=order_items["order_purchase_timestamp"].dt.month
order_items['year']=order_items["order_purchase_timestamp"].dt.year

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

gmv=order_items.groupby('year')['price'].sum()
print(gmv)
plt.figure(figsize=(5,5))
plt.plot(gmv.index,gmv.values)
plt.title("GMV by Year")
plt.xlabel("Year")
plt.ylabel("GMV")
plt.grid(True)
plt.xticks(gmv.index)
plt.show()

In [ ]:
# from google.cloud import bigquery

# client = bigquery.Client()


# # Example DataFrame
# df = pd.DataFrame({
#     "customer_id": [1, 2, 3],
#     "customer_name": ["Alice", "Bob", "Charlie"],
#     "spend": [100.5, 200.75, 300.0]
# })

# # Define target BigQuery details
# project_id = "dsai6mod2"
# dataset_id = "dsdwh"
# table_id = "sales_trends"

# # Option 1: Using pandas-gbq
# df.to_gbq(
#     destination_table=f"{dataset_id}.{table_id}",
#     project_id=project_id,
#     if_exists="replace"  # options: 'fail', 'replace', 'append'
# )


# print("DataFrame loaded successfully into BigQuery!")
